<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/ml-labs/LSATutorialWeek9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LSA Topic Modeling") \
    .getOrCreate()

## Load and Prepare the Data

In [2]:
# Sample data (a list of documents)
data = [
    (0, "Spark is a unified analytics engine for big data processing"),
    (1, "Machine learning is a method of data analysis that automates analytical model building"),
    (2, "Deep learning models are built using artificial neural networks"),
    (3, "Data science is an inter-disciplinary field that uses scientific methods, processes, algorithms, and systems to extract knowledge from data")
]

# Create DataFrame
df = spark.createDataFrame(data, ["id", "text"])
df.show(truncate=False)

+---+-------------------------------------------------------------------------------------------------------------------------------------------+
|id |text                                                                                                                                       |
+---+-------------------------------------------------------------------------------------------------------------------------------------------+
|0  |Spark is a unified analytics engine for big data processing                                                                                |
|1  |Machine learning is a method of data analysis that automates analytical model building                                                     |
|2  |Deep learning models are built using artificial neural networks                                                                            |
|3  |Data science is an inter-disciplinary field that uses scientific methods, processes, algorithms, and systems to extract

In [12]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StopWordsRemover
from pyspark.ml import Pipeline
from pyspark.mllib.linalg.distributed import RowMatrix
from pyspark.mllib.linalg import Vectors, Matrix
import numpy as np

## Text Preprocessing (Tokenization & Stopword Removal) Text Preprocessing Pipeline

In [9]:
# Text preprocessing pipeline
tokenizer = Tokenizer(inputCol="text", outputCol="words")
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[tokenizer, stopwords_remover, hashingTF, idf])

## Fit the Pipeline and Transform Data

In [10]:
# Fit and transform
model = pipeline.fit(df)
transformed_df = model.transform(df)

## Perform Singular Value Decomposition (SVD)

In [22]:
from pyspark.ml.feature import PCA

pca = PCA(k=2, inputCol="features", outputCol="reduced_features")
pca_model = pca.fit(transformed_df)
result = pca_model.transform(transformed_df)

In [23]:
# Show results
result.select("id", "text", "reduced_features").show(truncate=False)

# Save to HTML
html_output = """
<html>
<head><title>LSA Results</title></head>
<body>
<table border="1">
<tr><th>Document ID</th><th>Text</th><th>Reduced Features</th></tr>
"""

for row in result.collect():
    html_output += f"""
    <tr>
        <td>{row['id']}</td>
        <td>{row['text']}</td>
        <td>{row['reduced_features']}</td>
    </tr>
    """

html_output += """
</table>
</body>
</html>
"""

with open("lsa_results.html", "w") as file:
    file.write(html_output)

print("LSA results saved to lsa_results.html")

+---+-------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+
|id |text                                                                                                                                       |reduced_features                         |
+---+-------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+
|0  |Spark is a unified analytics engine for big data processing                                                                                |[-0.15815516172594524,0.6392472064131579]|
|1  |Machine learning is a method of data analysis that automates analytical model building                                                     |[-0.43702113735219617,1.3598575158163462]|
|2  |Deep learning models are built using artificial neural 

## Alternative Method to achieve this steps

In [11]:
# Convert to RDD of Vectors for RowMatrix
vectors = transformed_df.select("features").rdd.map(lambda row: Vectors.fromML(row.features))

In [14]:
# Create RowMatrix and compute SVD
row_matrix = RowMatrix(vectors)
svd = row_matrix.computeSVD(k=2, computeU=True)

In [16]:
# Get the reduced dimensions (U * Σ)
# Broadcast the singular values to the worker nodes
s_broadcast = spark.sparkContext.broadcast(svd.s.toArray())

reduced_features = svd.U.rows.map(lambda vector: vector.toArray() * s_broadcast.value).collect()

In [18]:
from pyspark.ml.linalg import DenseVector

# Add the reduced features back to the DataFrame
svd_result = transformed_df.rdd.zipWithIndex().map(lambda x: (x[1], x[0], DenseVector(reduced_features[x[1]]))).toDF(["index", "original", "svd_features"])

In [19]:
# Show results
svd_result.select("original.id", "original.text", "svd_features").show(truncate=False)

+---+-------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------+
|id |text                                                                                                                                       |svd_features                              |
+---+-------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------+
|0  |Spark is a unified analytics engine for big data processing                                                                                |[0.07134226371169068,0.006757190695360191]|
|1  |Machine learning is a method of data analysis that automates analytical model building                                                     |[0.09634414650557956,0.43131494148995236] |
|2  |Deep learning models are built using artificial ne